In [1]:
# =============================================================================
# STEP 1 — IMPORTS & CONFIG
# =============================================================================

import os
import json
import requests
import numpy as np
import pandas as pd
import rasterio
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# ── Paths (shared with goobie5000.ipynb outputs) ──────────────────────────────
OUT_DIR          = Path("./polk_corn_validation")
PRED_DIR         = OUT_DIR / "predictions"
PRED_RASTER_PATH = PRED_DIR / "scene_pred.tif"     # written by goobie5000 Step 6
YIELD_DIR        = OUT_DIR / "yield"
YIELD_DIR.mkdir(parents=True, exist_ok=True)

# ── Target region ─────────────────────────────────────────────────────────────
STATE_FIPS   = "19"       # Iowa
COUNTY_FIPS  = "153"      # Polk County
COUNTY_NAME  = "Polk"
STATE_NAME   = "IOWA"
TARGET_YEAR  = 2022

# ── Model config (matches goobie5000.ipynb Step 1) ────────────────────────────
CORN_CLASS_IDX  = 0
ACRES_PER_PIXEL = 900 / 4046.856   # 30m pixel → acres

print("✅ Config complete.")
print(f"   Target : {COUNTY_NAME} County, {STATE_NAME}  ({TARGET_YEAR})")
print(f"   Pred   : {PRED_RASTER_PATH}")


✅ Config complete.
   Target : Polk County, IOWA  (2022)
   Pred   : polk_corn_validation/predictions/scene_pred.tif


In [2]:
# =============================================================================
# STEP 2 — LOAD PREDICTED CORN MAP FROM GOOBIE5000
# =============================================================================
# Reads the scene_pred.tif written by Step 6 of goobie5000.ipynb.
# Computes total predicted corn acreage as the primary spatial feature.
# =============================================================================

if not PRED_RASTER_PATH.exists():
    raise FileNotFoundError(
        f"scene_pred.tif not found at {PRED_RASTER_PATH} Make sure goobie5000.ipynb Steps 5–6 have been run first.")

with rasterio.open(PRED_RASTER_PATH) as src:
    scene_pred      = src.read(1)    # (H, W) int16
    scene_transform = src.transform
    scene_crs       = src.crs

corn_pixels       = int(np.sum(scene_pred == CORN_CLASS_IDX))
total_valid_px    = int(np.sum(scene_pred >= 0))
pred_corn_acres   = corn_pixels * ACRES_PER_PIXEL
pred_corn_frac    = corn_pixels / max(total_valid_px, 1)

print(f"[OK] scene_pred loaded: {scene_pred.shape}  CRS: {scene_crs}")
print(f"     Corn pixels      : {corn_pixels:,}")
print(f"     Predicted acres  : {pred_corn_acres:,.0f} acres")
print(f"     Corn fraction    : {pred_corn_frac:.3f} ({pred_corn_frac*100:.1f}% of valid area)")


[OK] scene_pred loaded: (1422, 1350)  CRS: PROJCS["UTM Zone 15, Northern Hemisphere",GEOGCS["Unknown datum based upon the WGS 84 ellipsoid",DATUM["Not specified (based on WGS 84 spheroid)",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-93],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
     Corn pixels      : 323,776
     Predicted acres  : 72,006 acres
     Corn fraction    : 0.190 (19.0% of valid area)


In [3]:
# =============================================================================
# STEP 3 — USDA NASS CORN YIELD HISTORY
# =============================================================================
# Pulls county-level corn yield (bushels/acre) from the NASS QuickStats API.
# The dataset you linked covers Iowa county-level yield history.
#
# API docs: quickstats.nass.usda.gov/api
# Get a free API key at: quickstats.nass.usda.gov/api#param_define
# =============================================================================

NASS_API_KEY = os.environ.get("NASS_API_KEY", "AA30651F-5121-32B8-8E69-E52F9F356A04")

def fetch_nass_corn_yield(
    api_key:      str,
    state_name:   str  = STATE_NAME,
    county_name:  str  = COUNTY_NAME,
    start_year:   int  = 2000,
    end_year:     int  = TARGET_YEAR,
) -> pd.DataFrame:
    url    = "http://quickstats.nass.usda.gov/api/api_GET/"
    params = {
        "key":           api_key,
        "commodity_desc": "CORN",
        "statisticcat_desc": "YIELD",
        "unit_desc":     "BU / ACRE",
        "agg_level_desc": "COUNTY",
        "state_name":    state_name,
        "county_name":   county_name,
        "freq_desc":     "ANNUAL",
        "format":        "JSON",
    }
    print(f"   Fetching NASS yield data for {county_name} County, {state_name}...")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json().get("data", [])
    if not data:
        raise ValueError("No NASS data returned — check API key and parameters.")

    df = pd.DataFrame(data)
    df = df[["year", "Value", "county_name", "state_name"]].copy()
    df.columns = ["year", "yield_bu_acre", "county", "state"]
    df["year"]          = df["year"].astype(int)
    df["yield_bu_acre"] = pd.to_numeric(df["yield_bu_acre"].str.replace(",", ""), errors="coerce")
    df = df.dropna(subset=["yield_bu_acre"])
    df = df.sort_values("year").reset_index(drop=True)
    return df


yield_history = fetch_nass_corn_yield(NASS_API_KEY)

print(f"[OK] {len(yield_history)} years of yield data loaded")
print(f"{'Year':<8} {'Yield (bu/acre)':>16}")
print("  " + "-" * 26)
for _, row in yield_history.iterrows():
    marker = "  ← TARGET" if row["year"] == TARGET_YEAR else ""
    print(f"  {int(row['year']):<8} {row['yield_bu_acre']:>16.1f}{marker}")


   Fetching NASS yield data for Polk County, IOWA...
[OK] 99 years of yield data loaded
Year      Yield (bu/acre)
  --------------------------
  1926                 43.2
  1927                 39.4
  1928                 45.3
  1929                 41.0
  1930                 30.0
  1931                 43.1
  1932                 45.1
  1933                 44.2
  1934                 41.5
  1935                 41.5
  1936                 23.2
  1937                 48.8
  1938                 48.3
  1939                 56.8
  1940                 56.4
  1941                 55.8
  1942                 63.3
  1943                 60.1
  1944                 54.9
  1945                 52.5
  1946                 61.2
  1947                 27.1
  1948                 64.3
  1949                 50.2
  1950                 51.3
  1951                 46.2
  1952                 62.0
  1953                 55.8
  1954                 54.8
  1955                 53.3
  1956           

In [4]:
# =============================================================================
# STEP 4 — WEATHER & CLIMATE DATA (Open-Meteo — no API key required)
# =============================================================================
# Pulls daily weather for Polk County centroid for the TARGET_YEAR growing
# season (April–October). Computes agronomically meaningful features:
#   - GDD  : Growing Degree Days (base 50°F) — primary corn yield driver
#   - VPD  : Vapor Pressure Deficit — heat/drought stress indicator
#   - Precip: Total seasonal precipitation
#   - Tmax : Mean daily max temperature during pollination window (Jul)
# =============================================================================

# Polk County, Iowa centroid
POLK_LAT = 41.685
POLK_LON = -93.573

def fetch_weather(lat: float, lon: float, year: int) -> pd.DataFrame:
    url    = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": f"{year}-04-01",
        "end_date":   f"{year}-10-31",
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "et0_fao_evapotranspiration",
        ],
        "temperature_unit": "fahrenheit",
        "precipitation_unit": "inch",
        "timezone": "America/Chicago",
    }
    print(f"   Fetching weather for ({lat}, {lon})  {year}-04-01 → {year}-10-31...")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()["daily"]
    df   = pd.DataFrame(data)
    df.columns = ["date", "tmax_f", "tmin_f", "precip_in", "et0_in"]
    df["date"] = pd.to_datetime(df["date"])
    return df


def compute_weather_features(df: pd.DataFrame) -> dict:
    # Growing Degree Days (base 50°F, cap 86°F)
    df["tmax_cap"] = df["tmax_f"].clip(upper=86)
    df["tmean"]    = (df["tmax_cap"] + df["tmin_f"]) / 2
    df["gdd"]      = (df["tmean"] - 50).clip(lower=0)

    # Pollination stress window: July 1–31
    july_mask = df["date"].dt.month == 7
    tmax_july = df.loc[july_mask, "tmax_f"].mean()

    # Seasonal totals
    total_gdd    = df["gdd"].sum()
    total_precip = df["precip_in"].sum()
    total_et0    = df["et0_in"].sum()
    water_deficit = total_et0 - total_precip   # positive = drought stress

    return {
        "gdd_total":       round(total_gdd,    1),
        "precip_in":       round(total_precip, 2),
        "et0_in":          round(total_et0,    2),
        "water_deficit_in":round(water_deficit,2),
        "tmax_july_f":     round(tmax_july,    1),
    }


weather_df       = fetch_weather(POLK_LAT, POLK_LON, TARGET_YEAR)
weather_features = compute_weather_features(weather_df)

print(f"[OK] Weather features for {TARGET_YEAR} growing season:")
for k, v in weather_features.items():
    print(f"     {k:<25} : {v}")


   Fetching weather for (41.685, -93.573)  2022-04-01 → 2022-10-31...
[OK] Weather features for 2022 growing season:
     gdd_total                 : 3407.3
     precip_in                 : 26.04
     et0_in                    : 35.95
     water_deficit_in          : 9.91
     tmax_july_f               : 85.2


In [5]:
# =============================================================================
# STEP 5 — FEATURE ENGINEERING (FIXED)
# =============================================================================
# corn_frac is now fetched from NASS planted acreage for each historical year
# instead of using the 2022 scene_pred value for all years.
# =============================================================================

def fetch_nass_corn_acres_by_year(
    api_key:     str,
    state_name:  str = STATE_NAME,
    county_name: str = COUNTY_NAME,
) -> dict:
    """
    Fetches county-level corn PLANTED AREA (acres) from NASS for all available
    years. Returns a dict of {year: planted_acres}.
    Used to compute corn_frac per year instead of using the 2022 prediction
    for all years.
    """
    url    = "https://quickstats.nass.usda.gov/api/api_GET/"
    params = {
        "key":                api_key,
        "commodity_desc":     "CORN",
        "statisticcat_desc":  "AREA PLANTED",
        "unit_desc":          "ACRES",
        "agg_level_desc":     "COUNTY",
        "state_name":         state_name,
        "county_name":        county_name,
        "freq_desc":          "ANNUAL",
        "format":             "JSON",
    }
    print("   Fetching NASS historical corn planted acres by year...")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json().get("data", [])
    if not data:
        print("   ⚠️  No NASS planted area data found — corn_frac will be dropped.")
        return {}

    result = {}
    for row in data:
        try:
            yr    = int(row["year"])
            acres = float(row["Value"].replace(",", ""))
            result[yr] = acres
        except (ValueError, KeyError):
            continue

    print(f"   [OK] Planted acres found for {len(result)} years: "
          f"{min(result.keys())}–{max(result.keys())}")
    return result


def build_feature_matrix(
    yield_df:        pd.DataFrame,
    weather_year:    int,
    weather_feats:   dict,
    pred_corn_acres: float,
    pred_corn_frac:  float,
    nass_api_key:    str,
) -> pd.DataFrame:

    # ── Fetch historical planted acres for corn_frac per year ─────────────────
    # Total county area for Polk County, Iowa (fixed — 592 sq miles = 378,880 acres)
    POLK_TOTAL_ACRES = 378_880.0

    historical_acres = fetch_nass_corn_acres_by_year(nass_api_key)

    rows = []
    for _, row in yield_df.iterrows():
        yr = int(row["year"])

        # ── Weather features ──────────────────────────────────────────────────
        try:
            w_df   = fetch_weather(POLK_LAT, POLK_LON, yr)
            w_feat = compute_weather_features(w_df)
        except Exception as e:
            print(f"   ⚠️  Skipping {yr} — weather fetch failed: {e}")
            continue

        # ── corn_frac: use scene_pred for TARGET_YEAR, NASS acres for others ──
        if yr == weather_year:
            # Use your actual Prithvi model prediction for the target year
            corn_frac = pred_corn_frac
            corn_source = "scene_pred"
        elif yr in historical_acres:
            # Use NASS planted acres / total county acres for historical years
            corn_frac = historical_acres[yr] / POLK_TOTAL_ACRES
            corn_source = "NASS"
        else:
            # No spatial data available for this year — drop corn_frac
            corn_frac = np.nan
            corn_source = "missing"

        rows.append({
            "year":             yr,
            "yield_bu_acre":    row["yield_bu_acre"],
            "gdd_total":        w_feat["gdd_total"],
            "precip_in":        w_feat["precip_in"],
            "water_deficit_in": w_feat["water_deficit_in"],
            "tmax_july_f":      w_feat["tmax_july_f"],
            "corn_frac":        corn_frac,
            "corn_frac_source": corn_source,
        })

    df = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    return df


print("Building feature matrix (fetching historical weather — ~1–2 min)...")
feature_df = build_feature_matrix(
    yield_df        = yield_history,
    weather_year    = TARGET_YEAR,
    weather_feats   = weather_features,
    pred_corn_acres = pred_corn_acres,
    pred_corn_frac  = pred_corn_frac,
    nass_api_key    = NASS_API_KEY,
)

print(f"[OK] Feature matrix: {feature_df.shape}")
print(f"{'Year':<8} {'Yield':>8} {'GDD':>8} {'Precip':>8} "
      f"{'WaterDef':>10} {'TmaxJul':>9} {'CornFrac':>10} {'Source':>10}")
print("  " + "-" * 75)
for _, r in feature_df.iterrows():
    marker = " ← TARGET" if r["year"] == TARGET_YEAR else ""
    print(f"  {int(r['year']):<8} {r['yield_bu_acre']:>8.1f} "
          f"{r['gdd_total']:>8.0f} {r['precip_in']:>8.2f} "
          f"{r['water_deficit_in']:>10.2f} {r['tmax_july_f']:>9.1f} "
          f"{r['corn_frac']:>10.3f} {r['corn_frac_source']:>10}{marker}")


Building feature matrix (fetching historical weather — ~1–2 min)...
   Fetching NASS historical corn planted acres by year...
   [OK] Planted acres found for 99 years: 1926–2024
   Fetching weather for (41.685, -93.573)  1926-04-01 → 1926-10-31...
   ⚠️  Skipping 1926 — weather fetch failed: 400 Client Error: Bad Request for url: https://archive-api.open-meteo.com/v1/archive?latitude=41.685&longitude=-93.573&start_date=1926-04-01&end_date=1926-10-31&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=et0_fao_evapotranspiration&temperature_unit=fahrenheit&precipitation_unit=inch&timezone=America%2FChicago
   Fetching weather for (41.685, -93.573)  1927-04-01 → 1927-10-31...
   ⚠️  Skipping 1927 — weather fetch failed: 400 Client Error: Bad Request for url: https://archive-api.open-meteo.com/v1/archive?latitude=41.685&longitude=-93.573&start_date=1927-04-01&end_date=1927-10-31&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=et0_

In [6]:
# =============================================================================
# STEP 5.5 — FEATURE TRANSFORMATIONS
# =============================================================================
# Applied after build_feature_matrix() produces feature_df.
# Transformations are fit on training data only (years < TARGET_YEAR)
# to prevent data leakage into the target year row.
# =============================================================================

import numpy as np
import pandas as pd
from scipy import stats

# =============================================================================
# TRANSFORM 1 — YIELD DETRENDING (most important)
# Removes the long-term technology/genetics trend from yield so the model
# learns weather-driven variation, not year-over-year improvement.
# Stores the trend line so we can add it back to the final prediction.
# =============================================================================

train_mask = feature_df["year"] < TARGET_YEAR

# Fit linear trend on training years only
trend_x    = feature_df.loc[train_mask, "year"].values
trend_y    = feature_df.loc[train_mask, "yield_bu_acre"].values
slope, intercept, r_value, _, _ = stats.linregress(trend_x, trend_y)

print(f"Yield trend: {slope:.2f} bu/acre/year  "
      f"(R²={r_value**2:.3f})  intercept={intercept:.1f}")

# Apply detrending to ALL rows (including target year)
feature_df["yield_trend"]          = slope * feature_df["year"] + intercept
feature_df["yield_detrended"]      = feature_df["yield_bu_acre"] - feature_df["yield_trend"]

# Store trend params for adding back to prediction in Step 6
TREND_SLOPE     = slope
TREND_INTERCEPT = intercept

print(f"  Detrended yield range: "
      f"{feature_df.loc[train_mask,'yield_detrended'].min():.1f} to "
      f"{feature_df.loc[train_mask,'yield_detrended'].max():.1f} bu/acre")

# =============================================================================
# TRANSFORM 2 — LOG TRANSFORM PRECIPITATION
# Precipitation is right-skewed — a few very wet years dominate.
# Log transform compresses the high end and spreads the low end.
# Shift by +1 to handle near-zero values safely.
# =============================================================================

feature_df["precip_log"] = np.log1p(feature_df["precip_in"])

# =============================================================================
# TRANSFORM 3 — LAGGED YIELD (prior year's detrended yield)
# Prior year yield captures soil health, farmer confidence, and
# carryover effects. One of the strongest predictors in yield models.
# =============================================================================

feature_df = feature_df.sort_values("year").reset_index(drop=True)
feature_df["yield_lag1"] = feature_df["yield_detrended"].shift(1)

# The first year will have NaN lag — drop it from training only
# (target year lag is filled from the last known year)
target_year_idx = feature_df[feature_df["year"] == TARGET_YEAR].index
if not feature_df.loc[target_year_idx, "yield_lag1"].isna().any():
    pass   # lag available for target year — good
else:
    # Fill target year lag with last known detrended yield
    last_known = feature_df.loc[train_mask, "yield_detrended"].iloc[-1]
    feature_df.loc[target_year_idx, "yield_lag1"] = last_known
    print(f"  Target year lag filled with last known detrended yield: {last_known:.1f}")

# =============================================================================
# TRANSFORM 4 — GDD QUADRATIC TERM
# Corn yield has a non-linear response to heat accumulation:
# too little GDD = underdeveloped crop
# optimal GDD    = peak yield
# too much GDD   = heat stress, yield decline
# Adding GDD² lets the model capture this curve without needing a neural net.
# =============================================================================

# Normalize GDD first so the squared term doesn't dwarf other features
gdd_mean = feature_df.loc[train_mask, "gdd_total"].mean()
gdd_std  = feature_df.loc[train_mask, "gdd_total"].std()
feature_df["gdd_norm"]    = (feature_df["gdd_total"] - gdd_mean) / (gdd_std + 1e-9)
feature_df["gdd_norm_sq"] = feature_df["gdd_norm"] ** 2

# Store normalization params for applying to future years
GDD_MEAN = gdd_mean
GDD_STD  = gdd_std

# =============================================================================
# TRANSFORM 5 — WATER DEFICIT ASYMMETRY
# Excess water (negative deficit) has diminishing benefit above a threshold,
# while drought stress (positive deficit) has an accelerating penalty.
# Splitting into separate surplus/deficit features captures this asymmetry.
# =============================================================================

feature_df["water_surplus"] = (-feature_df["water_deficit_in"]).clip(lower=0)
feature_df["water_stress"]  = feature_df["water_deficit_in"].clip(lower=0)

# =============================================================================
# FINAL FEATURE SET
# =============================================================================

FEATURE_COLS = [
    "gdd_norm",          # normalized GDD
    "gdd_norm_sq",       # quadratic GDD term — captures heat stress curve
    "precip_log",        # log-transformed precipitation
    "water_surplus",     # excess water (beneficial, diminishing returns)
    "water_stress",      # drought stress (harmful, accelerating penalty)
    "tmax_july_f",       # July heat stress during pollination
    "corn_frac",         # spatial corn coverage (Prithvi for target, CDL for history)
    "yield_lag1",        # prior year detrended yield
]

# Use detrended yield as the target instead of raw yield
TARGET_COL = "yield_detrended"

# Drop rows missing lag (first year in dataset)
feature_df_model = feature_df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

print(f"[OK] Feature matrix ready: {feature_df_model.shape}")
print(f"     Features : {FEATURE_COLS}")
print(f"     Target   : {TARGET_COL} (detrended — trend will be added back in Step 6)")
print()

# Print transformed feature table
print(f"  {'Year':<8} {'Yield':>8} {'Detrnd':>8} {'GDDnrm':>8} "
      f"{'PrecipL':>8} {'WtrStr':>8} {'WtrSrp':>8} {'Lag1':>8} {'CornFrc':>8}")
print("  " + "-" * 75)
for _, r in feature_df_model.iterrows():
    marker = " ←" if r["year"] == TARGET_YEAR else ""
    print(f"  {int(r['year']):<8} {r['yield_bu_acre']:>8.1f} "
          f"{r['yield_detrended']:>8.1f} {r['gdd_norm']:>8.3f} "
          f"{r['precip_log']:>8.3f} {r['water_stress']:>8.2f} "
          f"{r['water_surplus']:>8.2f} {r['yield_lag1']:>8.1f} "
          f"{r['corn_frac']:>8.3f}{marker}")


Yield trend: 1.88 bu/acre/year  (R²=0.877)  intercept=-3619.0
  Detrended yield range: -45.9 to 27.8 bu/acre
[OK] Feature matrix ready: (84, 16)
     Features : ['gdd_norm', 'gdd_norm_sq', 'precip_log', 'water_surplus', 'water_stress', 'tmax_july_f', 'corn_frac', 'yield_lag1']
     Target   : yield_detrended (detrended — trend will be added back in Step 6)

  Year        Yield   Detrnd   GDDnrm  PrecipL   WtrStr   WtrSrp     Lag1  CornFrc
  ---------------------------------------------------------------------------
  1941         55.8     16.5   -0.100    3.191     9.09     0.00     19.0    0.242
  1942         63.3     22.1   -1.134    3.164     8.18     0.00     16.5    0.255
  1943         60.1     17.0   -1.611    3.160     9.41     0.00     22.1    0.290
  1944         54.9      9.9   -0.641    3.262     5.53     0.00     17.0    0.281
  1945         52.5      5.6   -2.210    3.286     5.21     0.00      9.9    0.301
  1946         61.2     12.5   -1.083    3.273     7.30     0.00

In [7]:
# =============================================================================
# STEP 6 — YIELD PREDICTION MODEL (XGBoost + RandomForest + Optimization)
# =============================================================================
# Uses TimeSeriesSplit CV throughout to prevent temporal data leakage.
# Hyperparameter search uses RandomizedSearchCV (better than GridSearch
# for small datasets — avoids overfitting the hyperparameter space).
# =============================================================================

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV,
    cross_val_score,
)
from sklearn.metrics import mean_absolute_error
from scipy.stats import randint, uniform

# ── Train/predict split ───────────────────────────────────────────────────────
train_df  = feature_df[feature_df["year"] < TARGET_YEAR].dropna(subset=FEATURE_COLS)
target_df = feature_df[feature_df["year"] == TARGET_YEAR]

if target_df.empty:
    raise ValueError(f"No features found for target year {TARGET_YEAR}.")

# Sort by year — critical for TimeSeriesSplit to work correctly
train_df  = feature_df_model[feature_df_model["year"] < TARGET_YEAR]
target_df = feature_df_model[feature_df_model["year"] == TARGET_YEAR]

X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET_COL].values      # ← detrended yield, not raw
X_pred  = target_df[FEATURE_COLS].values


# ── Scaling (XGBoost only — RandomForest is scale-invariant) ─────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_pred_sc  = scaler.transform(X_pred)

# ── TimeSeriesSplit — prevents future data leaking into past folds ────────────
# n_splits=5 means: train on years 1-4, test on 5 / train on 1-8, test on 9...
# min_train_size ensures each fold has enough data to be meaningful
n_splits = min(5, len(X_train) // 3)   # never more folds than data supports
tscv     = TimeSeriesSplit(n_splits=n_splits)

print(f"TimeSeriesSplit: {n_splits} folds  |  Training samples: {len(X_train)}")
print(f"Features       : {FEATURE_COLS}")
print()

# =============================================================================
# MODEL 1 — XGBoost with RandomizedSearchCV
# =============================================================================

xgb_param_dist = {
    "n_estimators":    randint(50, 300),
    "max_depth":       randint(2, 5),          # shallow — prevents overfitting
    "learning_rate":   uniform(0.01, 0.15),
    "subsample":       uniform(0.6, 0.4),      # 0.6–1.0
    "colsample_bytree":uniform(0.6, 0.4),      # 0.6–1.0
    "reg_alpha":       uniform(0.0, 0.5),      # L1
    "reg_lambda":      uniform(0.5, 2.0),      # L2
}

xgb_base = XGBRegressor(random_state=42, verbosity=0)

xgb_search = RandomizedSearchCV(
    estimator          = xgb_base,
    param_distributions= xgb_param_dist,
    n_iter             = 50,           # 50 random combos — enough for this space
    scoring            = "neg_mean_absolute_error",
    cv                 = tscv,         # ← TimeSeriesSplit, not random KFold
    random_state       = 42,
    n_jobs             = -1,
    refit              = True,         # refit best params on full training set
)

print("Optimizing XGBoost...")
xgb_search.fit(X_train_sc, y_train)
xgb_best_mae  = -xgb_search.best_score_
xgb_pred      = xgb_search.best_estimator_.predict(X_pred_sc)[0]
xgb_importances = dict(zip(FEATURE_COLS,
                            xgb_search.best_estimator_.feature_importances_))

print(f"  Best params : {xgb_search.best_params_}")
print(f"  CV MAE      : {xgb_best_mae:.1f} bu/acre")
print(f"  Prediction  : {xgb_pred:.1f} bu/acre")

# =============================================================================
# MODEL 2 — RandomForest with RandomizedSearchCV
# =============================================================================

rf_param_dist = {
    "n_estimators":   randint(100, 600),
    "max_depth":      randint(2, 6),
    "min_samples_leaf":randint(2, 6),      # higher = more regularization
    "min_samples_split":randint(2, 8),
    "max_features":   ["sqrt", "log2", 0.5, 0.7],
}

rf_base = RandomForestRegressor(random_state=42)

rf_search = RandomizedSearchCV(
    estimator          = rf_base,
    param_distributions= rf_param_dist,
    n_iter             = 50,
    scoring            = "neg_mean_absolute_error",
    cv                 = tscv,         # ← TimeSeriesSplit here too
    random_state       = 42,
    n_jobs             = -1,
    refit              = True,
)

print("Optimizing RandomForest...")
rf_search.fit(X_train, y_train)
rf_best_mae   = -rf_search.best_score_
rf_pred       = rf_search.best_estimator_.predict(X_pred)[0]
rf_importances = dict(zip(FEATURE_COLS,
                           rf_search.best_estimator_.feature_importances_))

print(f"  Best params : {rf_search.best_params_}")
print(f"  CV MAE      : {rf_best_mae:.1f} bu/acre")
print(f"  Prediction  : {rf_pred:.1f} bu/acre")

# =============================================================================
# MODEL SELECTION + ENSEMBLE
# =============================================================================

# ── Weighted ensemble — blend both predictions weighted by inverse MAE ────────
# Better than hard-picking one model on small datasets since both carry signal
xgb_weight    = 1 / (xgb_best_mae + 1e-9)
rf_weight     = 1 / (rf_best_mae  + 1e-9)
total_weight  = xgb_weight + rf_weight
ensemble_pred = (xgb_pred * xgb_weight + rf_pred * rf_weight) / total_weight

# ── Pick best single model for feature importance reporting ───────────────────
if xgb_best_mae <= rf_best_mae:
    best_name    = "XGBoost"
    importances  = xgb_importances
    best_mae     = xgb_best_mae
else:
    best_name    = "RandomForest"
    importances  = rf_importances
    best_mae     = rf_best_mae

# Use ensemble as final prediction
predicted_yield = ensemble_pred

# =============================================================================
# RESULTS
# =============================================================================

# ══ FIX 2 — ADD TREND BACK ════════════════════════════════════════════════════
# TREND_SLOPE and TREND_INTERCEPT were set in Step 5.5.
# predicted_yield is a detrended residual (e.g. +8.3 means 8.3 above trend).
# Add the trend value for TARGET_YEAR to get final bushels/acre.
trend_for_target_year = TREND_SLOPE * TARGET_YEAR + TREND_INTERCEPT
predicted_yield_raw   = predicted_yield + trend_for_target_year
# ══════════════════════════════════════════════════════════════════════════════

# ── Results block — replace predicted_yield with predicted_yield_raw ──────────
print(f"  Detrended prediction            : {predicted_yield:+.1f} bu/acre (vs. trend)")
print(f"  Trend for {TARGET_YEAR}               : {trend_for_target_year:.1f} bu/acre")
print(f"  Final yield prediction          : {predicted_yield_raw:.1f} bu/acre")

actual_row = yield_history[yield_history["year"] == TARGET_YEAR]
if not actual_row.empty:
    actual_yield = actual_row["yield_bu_acre"].values[0]
    error        = abs(predicted_yield_raw - actual_yield)   # ← use raw here
    pct_error    = error / actual_yield * 100
    print(f"  Actual NASS yield ({TARGET_YEAR})    : {actual_yield:.1f} bu/acre")
    print(f"  Absolute error                  : {error:.1f} bu/acre")
    print(f"  Relative error                  : {pct_error:.1f}%")
    if pct_error <= 5:
        print("  ✅ Excellent — within 5% of NASS ground truth")
    elif pct_error <= 10:
        print("  ⚠️  Good — within 10% of NASS ground truth")
    else:
        print("  ❌ High error — consider adding more features or years")

# ── Use predicted_yield_raw for all acreage/production calculations ───────────
print(f"  Predicted corn acres            : {pred_corn_acres:,.0f} acres")
print(f"  Estimated total production      : {predicted_yield_raw * pred_corn_acres:,.0f} bushels")
print(sep)




TimeSeriesSplit: 5 folds  |  Training samples: 81
Features       : ['gdd_norm', 'gdd_norm_sq', 'precip_log', 'water_surplus', 'water_stress', 'tmax_july_f', 'corn_frac', 'yield_lag1']

Optimizing XGBoost...
  Best params : {'colsample_bytree': np.float64(0.7024273291045295), 'learning_rate': np.float64(0.016065038430764702), 'max_depth': 4, 'n_estimators': 64, 'reg_alpha': np.float64(0.05544541040591566), 'reg_lambda': np.float64(1.3786730037315402), 'subsample': np.float64(0.6806876809341584)}
  CV MAE      : 13.7 bu/acre
  Prediction  : 1.0 bu/acre
Optimizing RandomForest...
  Best params : {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 3, 'n_estimators': 317}
  CV MAE      : 13.7 bu/acre
  Prediction  : 3.1 bu/acre
  Detrended prediction            : +2.1 bu/acre (vs. trend)
  Trend for 2022               : 192.0 bu/acre
  Final yield prediction          : 194.0 bu/acre
  Actual NASS yield (2022)    : 188.6 bu/acre
  Absolute error              

NameError: name 'sep' is not defined

In [ ]:
# =============================================================================
# STEP 7 — SAVE RESULTS
# =============================================================================

results = {
    "county":          f"{COUNTY_NAME} County, {STATE_NAME}",
    "year":            TARGET_YEAR,
    "predicted_yield": round(predicted_yield, 2),
    "pred_corn_acres": round(pred_corn_acres, 1),
    "est_production":  round(predicted_yield * pred_corn_acres, 0),
    "cv_mae_bu_acre":  round(cv_mae, 2),
    "weather":         weather_features,
    "feature_importance": {k: round(v, 4) for k, v in importances.items()},
}

if not actual_row.empty:
    results["actual_yield"]    = float(actual_yield)
    results["absolute_error"]  = round(float(error), 2)
    results["relative_error_pct"] = round(float(error / actual_yield * 100), 2)

out_path = YIELD_DIR / f"yield_prediction_{TARGET_YEAR}.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved: {out_path}")
print("🌽 Done!")
